# 02 · Metadata & Reproducibility

An MDC is only useful if others can **trust** and **reproduce** it. gwmock
writes a metadata sidecar for every run and can re-run a simulation from
that metadata alone. We'll prove the re-run is bit-for-bit identical.

<!-- Replace USER/REPO/BRANCH after pushing; Isaac will wire the real Colab links. -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USER/REPO/blob/main/materials/notebooks/02-metadata-reproducibility.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in correlated-noise support used in notebook 04.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

## 1. Make a small reproducible run

We use the bundled **SGWB** example here because its signal is written as
HDF5 — handy for an exact byte comparison later (more on SGWB in nb 04).

In [ ]:
!gwmock config --get signal/sgwb/et_2l_aligned --output config.yaml && cat config.yaml

In [ ]:
!gwmock simulate config.yaml --author 'Workshop' --email 'you@example.org'

## 2. Inside the metadata

One JSON file captures the **entire provenance** of the run.

In [ ]:
import json, glob
meta_path = sorted(glob.glob('metadata/*.metadata.json'))[0]
meta = json.load(open(meta_path))
print(meta_path, '\n')
print('top-level keys:')
print('  ', ', '.join(meta.keys()))

In [ ]:
# The provenance that makes a run reproducible & auditable:
print('gwmock version :', meta['gwmock_version'])
print('subpackages    :', meta['subpackage_versions'])
print('host           :', meta['host'])
print('root seed      :', meta['seed'])
print('segment seeds  :', meta['segment_seeds'])
print('config sha256  :', meta['config_sha256'])
print()
print('per-output records:')
for o in meta['outputs']:
    print('  %-8s %s' % (o['kind'], o['path']))
    print('           sha256=%s  t0=%s dur=%s' % (o['sha256'][:23]+'...', o['t0'], o['duration']))

Note what's recorded: **package versions**, the **host**, the **root seed**
and the **derived per-segment seeds**, a **hash of the config**, and a
**sha256 of every output file**. That is everything needed to reproduce
and to detect tampering or corruption.

## 3. Reproduce from metadata alone

`gwmock simulate <metadata.json>` ignores nothing and re-derives the run
from the stored config + seeds. We point it at a clean directory.

In [ ]:
import shutil, os
shutil.rmtree('repro', ignore_errors=True); os.makedirs('repro')
shutil.copy(meta_path, 'repro/')

In [ ]:
!cd repro && gwmock simulate *.metadata.json 2>&1 | tail -3

## 4. Did we get the *same data* back?

Two levels of checking:
1. **The science** — the strain arrays themselves.
2. **The bytes** — the file hashes.

In [ ]:
import h5py, numpy as np
orig = 'output/signal/sgwb-et-2l-aligned-0.hdf5'
repr_ = 'repro/output/signal/sgwb-et-2l-aligned-0.hdf5'
with h5py.File(orig) as a, h5py.File(repr_) as b:
    ds = list(a.keys())[0]
    same = np.array_equal(a[ds][:], b[ds][:])
print('SGWB signal arrays identical :', same)

from gwpy.timeseries import TimeSeries
ch = 'ET1_2L_ALIGNED_SARD:NOISE'
na = TimeSeries.read('output/noise/noise-0-ET1_2L_ALIGNED_SARD.gwf', channel=ch).value
nb = TimeSeries.read('repro/output/noise/noise-0-ET1_2L_ALIGNED_SARD.gwf', channel=ch).value
print('noise strain arrays identical:', np.array_equal(na, nb))

**The science is exactly reproducible** — same seed in, same strain out,
on any machine with the same package versions.

### A reproducibility subtlety worth knowing

Compare the *raw file* hashes and you'll see the HDF5 signal matches but
the **GWF noise files differ**:

In [ ]:
import hashlib
def sha(p):
    return hashlib.sha256(open(p,'rb').read()).hexdigest()[:16]
for rel in ['signal/sgwb-et-2l-aligned-0.hdf5', 'noise/noise-0-ET1_2L_ALIGNED_SARD.gwf']:
    a, b = sha('output/'+rel), sha('repro/output/'+rel)
    print('%-45s %s  %s  %s' % (rel, a, b, 'MATCH' if a==b else 'differ'))

> The **GWF container** embeds a write-time timestamp in its header, so two
> byte-identical strains written at different wall-clock times hash
> differently. The HDF5 signal has no such stamp and matches exactly.
> **Lesson:** for reproducibility, compare the *decoded data* (as above),
> not the raw container bytes.

## 5. `gwmock validate` — integrity against recorded hashes

`validate` recomputes each output's sha256 and compares it to the metadata.
It's the right tool to check a *downloaded* dataset arrived intact.

In [ ]:
!gwmock validate output/ --metadata-paths metadata/

> ⚠️ **Known issue (report it / check your version).** On current releases
> `validate` may list each file twice — once as `PASS` and once as
> `File not found` — because it reconstructs output paths without their
> sub-directory. The `PASS` rows are the real result. A fix is on the way;
> if you see only `PASS` rows, you're on a patched build.

➡️ Next: `03-downstream-integration.ipynb`.